# Description
We aim to compare the process of creating a PD model using a logistic regression and a common ML method (LGBM - Ligh Gradient Boosting Machine)
This notebook will train the ML model LGBM (Light Gradient Boosting Machine) this is an algorithm from microsoft which uses tree based learning algorithms (like XGBoost), however it is known to be more efficient, faster and more accurate
<br>[lgbm website](https://lightgbm.readthedocs.io/en/latest/index.html)

# Setup

In [25]:
import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import Image
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn import tree
import statsmodels.api as sm
import joblib
import lightgbm as lgbm
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import ParameterSampler, KFold

# Data

In [26]:
project_folder = "02_base_pipeline"

input_path = f"..\\..\\data\\inputs\\{project_folder}\\"
output_path = f"..\\..\\data\\outputs\\{project_folder}\\"

In [27]:
train_df = pd.read_parquet(f'{output_path}train_df.parquet')
test_df =  pd.read_parquet(f'{output_path}test_df.parquet')

# Analysis

In [65]:
x_cols = \
['loan_amnt', 
 'funded_amnt', 
 'funded_amnt_inv', 
 'term', 
 'int_rate',
 'installment', 
 'grade', 
 'sub_grade', 
 'emp_length', 
 'home_ownership',
 'annual_inc', 
 'verification_status', 
 'pymnt_plan',
 'purpose', 
 'addr_state', 
 'dti', 
 'delinq_2yrs', 
 'fico_range_low',
 'fico_range_high', 
 'inq_last_6mths',
 'open_acc', 
 'pub_rec', 
 'revol_bal',
 'revol_util', 
 'total_acc', 
 'initial_list_status', 
#  'out_prncp',
#  'out_prncp_inv', 
#  'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp',
# 'total_rec_int', 'total_rec_late_fee', 'last_pymnt_amnt', 'last_fico_range_high',
# 'last_fico_range_low', 'policy_code', 'application_type',
# 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim',
# 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util',
# 'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct',
# 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl',
# 'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_inq',
# 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl',
# 'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl',
# 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m',
# 'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m',
# 'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec_bankruptcies',
# 'tax_liens', 'tot_hi_cred_lim', 'total_bal_ex_mort', 'total_bc_limit',
# 'total_il_high_credit_limit', 'hardship_flag', 'disbursement_method',
# 'debt_settlement_flag'
]


y_col = 'default_flag'


def objectToCategory(df: pd.DataFrame):
    cols_dict = df.dtypes.to_dict()
    change_type_dict = {}
    for col_i, type_i in cols_dict.items():
        if type_i == "object":
            change_type_dict[col_i] = "category"
    result = df.astype(change_type_dict)
    return result

train_df_ = train_df\
    .pipe(objectToCategory)\
    .sample(frac=0.20, random_state=42)

Xtrain = train_df_[x_cols]
ytrain = train_df_[y_col]

test_df_ = test_df.pipe(objectToCategory)
Xtest = test_df_[x_cols]
ytest = test_df_[y_col]

## Feature Selection
For the LGBM model we will be applying a sequential feature elimination approach, to do it we will train a first model with all possible candidates, do a first round of hyperparameters optimisation, this round is just to get a baseline model for the feature selection step, will not be our final hyperparameters, finally, we will do the SFE (Sequential Feature Elmination).
<br> To do the SFE, we will sequentially eliminate features out of the model, each round will eliminate one feature at a time and check how the performance (in AUC/Gini) is affected, the feature that has the lowest impact in the performance will be eliminated. This will be repeated until the model has just one feature.
<br> This will allow us to verify how much performance each feature is adding in a sequential way and define a threshold where adding a new feature will not bring best performance to the model.

In [66]:
class RandomizedSearchHyperParams:
    def __init__(self,
                 model,
                 params_dist,
                 n_iter,
                 random_state,
                 cv):
        self.model = model
        self.params_dist = params_dist
        self.n_iter = n_iter
        self.random_state = random_state
        self.cv = cv
    
    def get_eval_sets(self, X, y, eval_set):
        kf = KFold(n_splits=self.cv, random_state=self.random_state, shuffle=True)
        kf.get_n_splits(X)
        splits = list(kf.split(X))
        train_dict = {}
        if eval_set is not None:
            train_dict[0] = eval_set
        else:
            for cv_i, split in enumerate(splits):
                train_dict[cv_i]=[]
                for slice in split:
                    train_dict[cv_i].append((X.iloc[slice], y.iloc[slice]))
        return train_dict
    
    def fit(self, X, y, eval_set = None):
        train_dict = self.get_eval_sets(X, y, eval_set)    
        results = []
        self.params_list = list(ParameterSampler(self.params_dist, self.n_iter, random_state=self.random_state))
        for i, params_i in enumerate(self.params_list):
            for cv_i, eval_set in train_dict.items():
                model_i = self.model(**params_i)
                model_i.fit(eval_set[0][0], eval_set[0][1], eval_set=eval_set, eval_names=['train', 'test'], eval_metric='auc')
                result_i= {
                    'n_iter': i,
                    'n_cv' : cv_i,
                    'model' : model_i,
                    'train_auc' : model_i.best_score_['train']['auc'],
                    'test_auc' : model_i.best_score_['test']['auc'],
                    'train_gini' : model_i.best_score_['train']['auc']*2-1,
                    'test_gini' : model_i.best_score_['test']['auc']*2-1,
                    'overfit' : np.abs((model_i.best_score_['train']['auc']*2-1) - (model_i.best_score_['test']['auc']*2-1)),
                    'score' : (model_i.best_score_['test']['auc']*2-1) - np.abs((model_i.best_score_['train']['auc']*2-1) -(model_i.best_score_['test']['auc']*2-1))
                }
                for param_name, param_selected in params_i.items():
                    result_i[param_name] = param_selected
                results.append(result_i)
        results_df = pd.DataFrame(results)
        results_df['score_mean'] = results_df.groupby('n_iter')['score'].transform('mean')
        results_df['score_std'] = results_df.groupby('n_iter')['score'].transform('std').fillna(0)
        results_df['score_mean_std_adj'] = results_df['score_mean'] - results_df['score_std']
        self.results_df = results_df
        self.selected_params = results_df[results_df['score_mean_std_adj']==results_df['score_mean_std_adj'].max()]['model'].iloc[0].get_params()
        final_model = self.model(**self.selected_params)
        final_model.fit(X,y,eval_set=[(X,y)], eval_names=['train'], eval_metric='auc')
        self.best_estimator_ = final_model


class sequentialFeatureElimination:
    def __init__(self,
                 params,
                 train_df: pd.DataFrame,
                 features_list: list,
                 target: str,
                 model = lgbm.LGBMClassifier,
                 feature_in_gini_threshold: float = 0.0001,
                 ):
        self.params = params
        self.train_df, self.holdout_df = train_test_split(train_df, train_size=0.5, random_state=42)
        self.Xtrain = self.train_df[features_list]
        self.ytrain = self.train_df[target]
        self.Xholdout = self.holdout_df[features_list]
        self.yholdout = self.holdout_df[target]
        self.model = model
        self.features_list = features_list
        self.features_selected = features_list
        self.feature_in_gini_threshold = feature_in_gini_threshold
        self.hyperParamsOptResults = {}
    
    def hyperparametersOptimisation(self, n_iter: int =50):
        i = len(self.hyperParamsOptResults)
        clf = RandomizedSearchHyperParams(self.model, self.params, n_iter=n_iter, random_state=42, cv=2)
        Xtrain_ = self.Xtrain[self.features_selected]
        Xholdout_ = self.Xholdout[self.features_selected]
        clf.fit(Xtrain_, self.ytrain, eval_set = [(Xtrain_, self.ytrain), (Xholdout_, self.yholdout)])
        self.hyperParamsOptResults[i] = clf
        self.selectedModel = clf.best_estimator_
        self.feature_params_select = clf.best_estimator_.get_params()
        self.hparamsRan = True
    
    def featureSelection(self):
        results = []
        dropped_features = []
        j = 0
        if not self.hparamsRan:
            print("Running Initial Params opt first")
            self.hyperparametersOptimisation()
        while len(self.features_selected) >= 2:
            for col_i in self.features_selected:
                features_in = [i for i in self.features_selected if i != col_i]
                model_opt_features = self.model(**self.feature_params_select)
                Xtrain_ = self.Xtrain[features_in]
                Xholdout_ = self.Xholdout[features_in]
                model_opt_features.fit(Xtrain_, self.ytrain, eval_set = [(Xholdout_, self.yholdout)], eval_names=["holdout"], eval_metric='auc')
                results.append({'round': j,
                                'col_removed': col_i,
                                'features_in': features_in,
                                'model':model_opt_features, 
                                'auc': model_opt_features.best_score_['holdout']['auc']})
            result_auc = pd.DataFrame(results)
            result_auc = result_auc[result_auc['round']==j]
            dropped_features.append(result_auc.sort_values(['auc'], ascending=False)['col_removed'].iloc[0])
            self.features_selected = [i for i in self.features_list if i not in dropped_features]
            j = j + 1
        results_df = pd.DataFrame(results)
        results_df['gini'] = results_df['auc']*2 - 1
        results_df['max_auc'] = results_df.groupby(['round'])['auc'].transform('max')
        results_df = results_df[results_df['auc']==results_df['max_auc']]
        results_df['n_features'] = results_df['round'].max() - results_df['round'] + 1
        results_df = results_df.drop_duplicates(subset=['round', 'auc'], keep='first')
        results_df = results_df.sort_values('n_features')
        results_df['gini_impact'] = results_df['gini'].diff().fillna(results_df['gini'])
        results_df['gini_next5'] = results_df['gini_impact'].rolling(6).sum() - results_df['gini_impact']
        results_df['n_selected_features'] = np.where(results_df['gini_next5'] <= self.feature_in_gini_threshold, results_df['n_features'], None)
        results_df['n_selected_features'] = results_df['n_selected_features'].min()
        self.results_df = results_df
        self.results = results
        self.features_selected = results_df[results_df['n_features'] == results_df['n_selected_features']]['features_in'].iloc[0]
        

In [68]:
params = {'boosting_type': ['gbdt'],
          'is_unbalance' : [True, False],
          'colsample_bytree': [0.2, 0.5, 1.0],
          'importance_type': ['split'],
          'learning_rate': [0.01, 0.05, 0.1],
          'max_depth': [5, 10, 15],
          'min_child_samples': [1, 20, 30, 40],
          'min_child_weight': [0.001],
          'min_split_gain': [0.0],
          'n_estimators': [100],
          'num_leaves': [10, 31, 100],
          'objective': ['binary'],
          'random_state': [42],
          'reg_alpha': [0.0, 0.01, 0.05],
          'reg_lambda': [0.0, 0.01, 0.05],
          'subsample': [0.6, 0.8, 1.0],
          'subsample_for_bin': [100, 500, 2000],
          'subsample_freq': [0]}

SFE = sequentialFeatureElimination(
    params=params,
    train_df=train_df_,
    features_list=x_cols,
    target=y_col,
    model=lgbm.LGBMClassifier,
    feature_in_gini_threshold = 0.0001
)

SFE.hyperparametersOptimisation(n_iter = 150)

[LightGBM] [Warning] Using too small ``bin_construct_sample_cnt`` may encounter unexpected errors and poor accuracy.
[LightGBM] [Info] Number of positive: 12537, number of negative: 49443
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004569 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 441
[LightGBM] [Info] Number of data points in the train set: 61980, number of used features: 26
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.202275 -> initscore=-1.372136
[LightGBM] [Info] Start training from score -1.372136
[LightGBM] [Warning] Using too small ``bin_construct_sample_cnt`` may encounter unexpected errors and poor accuracy.
[LightGBM] [Info] Number of positive: 12537, number of negative: 49443
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002745 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2516
[Light

In [69]:
SFE.selectedModel.get_params()

{'boosting_type': 'gbdt',
 'class_weight': None,
 'colsample_bytree': 0.2,
 'importance_type': 'split',
 'learning_rate': 0.05,
 'max_depth': 10,
 'min_child_samples': 1,
 'min_child_weight': 0.001,
 'min_split_gain': 0.0,
 'n_estimators': 100,
 'n_jobs': None,
 'num_leaves': 10,
 'objective': 'binary',
 'random_state': 42,
 'reg_alpha': 0.01,
 'reg_lambda': 0.0,
 'subsample': 0.8,
 'subsample_for_bin': 100,
 'subsample_freq': 0,
 'is_unbalance': False}

In [70]:
SFE.featureSelection()

[LightGBM] [Warning] Using too small ``bin_construct_sample_cnt`` may encounter unexpected errors and poor accuracy.
[LightGBM] [Info] Number of positive: 12537, number of negative: 49443
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009660 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 415
[LightGBM] [Info] Number of data points in the train set: 61980, number of used features: 25
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.202275 -> initscore=-1.372136
[LightGBM] [Info] Start training from score -1.372136
[LightGBM] [Warning] Using too small ``bin_construct_sample_cnt`` may encounter unexpected errors and poor accuracy.
[LightGBM] [Info] Number of positive: 12537, number of negative: 49443
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002473 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set

In [71]:
SFE.results_df

,round,col_removed,features_in,model,auc,gini,max_auc,n_features,gini_impact,gini_next5,n_selected_features
349,24,dti,[grade],"LGBMClassifier(colsample_bytree=0.2, is_unbala...",0.677359,0.354717,0.677359,1,0.354717,NaN,20
346,23,home_ownership,"[grade, dti]","LGBMClassifier(colsample_bytree=0.2, is_unbala...",0.688639,0.377278,0.688639,2,0.022560,NaN,20
343,22,verification_status,"[grade, home_ownership, dti]","LGBMClassifier(colsample_bytree=0.2, is_unbala...",0.692955,0.385910,0.692955,3,0.008632,NaN,20
340,21,fico_range_low,"[grade, home_ownership, verification_status, dti]","LGBMClassifier(colsample_bytree=0.2, is_unbala...",0.694479,0.388957,0.694479,4,0.003047,NaN,20
331,20,emp_length,"[grade, home_ownership, verification_status, d...","LGBMClassifier(colsample_bytree=0.2, is_unbala...",0.695477,0.390955,0.695477,5,0.001997,NaN,20
329,19,open_acc,"[grade, emp_length, home_ownership, verificati...","LGBMClassifier(colsample_bytree=0.2, is_unbala...",0.692823,0.385647,0.692823,6,-0.005308,0.390955,20
322,18,total_acc,"[grade, emp_length, home_ownership, verificati...","LGBMClassifier(colsample_bytree=0.2, is_unbala...",0.697416,0.394832,0.697416,7,0.009185,0.030929,20
306,17,term,"[grade, emp_length, home_ownership, verificati...","LGBMClassifier(colsample_bytree=0.2, is_unbala...",0.698541,0.397083,0.698541,8,0.002251,0.017554,20
300,16,annual_inc,"[term, grade, emp_length, home_ownership, veri...","LGBMClassifier(colsample_bytree=0.2, is_unbala...",0.701779,0.403557,0.701779,9,0.006475,0.011172,20
293,15,inq_last_6mths,"[term, grade, emp_length, home_ownership, annu...","LGBMClassifier(colsample_bytree=0.2, is_unbala...",0.702088,0.404177,0.702088,10,0.000619,0.014600,20


In [72]:
results_df = SFE.results_df.sort_values('round')
fig = px.scatter(results_df,
                 x='n_features',
                 y='gini',
                 width=500,
                 height=300,
                 template='none')
fig.update_yaxes(tickformat=".1%")

In [73]:
fig = px.scatter(results_df,
                 x='n_features',
                 y='gini_impact',
                 width=500,
                 height=300,
                 template='none')
fig.update_yaxes(tickformat=".1%")

In [74]:
SFE.features_selected

['funded_amnt',
 'funded_amnt_inv',
 'term',
 'installment',
 'grade',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'fico_range_low',
 'inq_last_6mths',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status']

## Final Hyperparameters Opt

In [75]:
x_cols = SFE.features_selected


Xtrain = train_df_.pipe(objectToCategory)[x_cols]
ytrain = train_df_.pipe(objectToCategory)[y_col]

Xtest = test_df.pipe(objectToCategory)[x_cols]
ytest = test_df.pipe(objectToCategory)[y_col]

In [76]:
params = {'boosting_type': ['gbdt'],
          'is_unbalance' : [True, False],
          'colsample_bytree': [0.2, 0.5, 1.0],
          'importance_type': ['split'],
          'learning_rate': [0.01, 0.05, 0.1],
          'max_depth': [5, 10, 15],
          'min_child_samples': [1, 20, 30, 40],
          'min_child_weight': [0.001],
          'min_split_gain': [0.0],
          'n_estimators': [100],
          'num_leaves': [10, 31, 100],
          'objective': ['binary'],
          'random_state': [42],
          'reg_alpha': [0.0, 0.01, 0.05],
          'reg_lambda': [0.0, 0.01, 0.05],
          'subsample': [0.6, 0.8, 1.0],
          'subsample_for_bin': [100, 500, 2000],
          'subsample_freq': [0],
          'n_jobs': [-1],
          'random_state' : [42]}

lgbm_model_opt = lgbm.LGBMClassifier
clf = RandomizedSearchHyperParams(lgbm_model_opt, params, 150, random_state=42, cv=3)
# clf = RandomizedSearchCV(lgbm_model_opt, params, n_iter=150, random_state=42, scoring="neg_loss_score", cv=2, verbose=1)
clf.fit(Xtrain, ytrain)

[LightGBM] [Warning] Using too small ``bin_construct_sample_cnt`` may encounter unexpected errors and poor accuracy.
[LightGBM] [Info] Number of positive: 16738, number of negative: 65902
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003847 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 337
[LightGBM] [Info] Number of data points in the train set: 82640, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.202541 -> initscore=-1.370487
[LightGBM] [Info] Start training from score -1.370487
[LightGBM] [Warning] Using too small ``bin_construct_sample_cnt`` may encounter unexpected errors and poor accuracy.
[LightGBM] [Info] Number of positive: 17021, number of negative: 65619
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003586 seconds.
You can set `force_row_wise=true` 

In [77]:
results_df = clf.results_df
results_df.sort_values('score_mean_std_adj', ascending=False)

,n_iter,n_cv,model,train_auc,test_auc,train_gini,test_gini,overfit,score,subsample_freq,...,min_child_samples,max_depth,learning_rate,is_unbalance,importance_type,colsample_bytree,boosting_type,score_mean,score_std,score_mean_std_adj
69,23,0,"LGBMClassifier(colsample_bytree=0.5, is_unbala...",0.710136,0.701193,0.420272,0.402385,0.017887,0.384498,0,...,20,10,0.01,False,split,0.5,gbdt,0.399282,0.013673,0.385608
70,23,1,"LGBMClassifier(colsample_bytree=0.5, is_unbala...",0.705737,0.710678,0.411474,0.421356,0.009882,0.411474,0,...,20,10,0.01,False,split,0.5,gbdt,0.399282,0.013673,0.385608
71,23,2,"LGBMClassifier(colsample_bytree=0.5, is_unbala...",0.709059,0.704998,0.418119,0.409995,0.008123,0.401872,0,...,20,10,0.01,False,split,0.5,gbdt,0.399282,0.013673,0.385608
225,75,0,"LGBMClassifier(colsample_bytree=0.5, is_unbala...",0.712103,0.702028,0.424207,0.404055,0.020151,0.383904,0,...,20,5,0.01,False,split,0.5,gbdt,0.399618,0.015722,0.383896
227,75,2,"LGBMClassifier(colsample_bytree=0.5, is_unbala...",0.710733,0.705267,0.421467,0.410535,0.010932,0.399603,0,...,20,5,0.01,False,split,0.5,gbdt,0.399618,0.015722,0.383896
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
292,97,1,"LGBMClassifier(is_unbalance=False, max_depth=1...",0.859733,0.710932,0.719467,0.421864,0.297602,0.124262,0,...,30,10,0.10,False,split,1.0,gbdt,0.114179,0.009270,0.104909
293,97,2,"LGBMClassifier(is_unbalance=False, max_depth=1...",0.859640,0.707881,0.719279,0.415763,0.303516,0.112247,0,...,30,10,0.10,False,split,1.0,gbdt,0.114179,0.009270,0.104909
146,48,2,"LGBMClassifier(is_unbalance=True, max_depth=15...",0.870531,0.703678,0.741062,0.407356,0.333706,0.073651,0,...,1,15,0.10,True,split,1.0,gbdt,0.074547,0.010921,0.063626
145,48,1,"LGBMClassifier(is_unbalance=True, max_depth=15...",0.870548,0.706746,0.741097,0.413492,0.327604,0.085888,0,...,1,15,0.10,True,split,1.0,gbdt,0.074547,0.010921,0.063626


## Final Model

In [78]:
x_cols = SFE.features_selected
params = clf.selected_params
print(x_cols)
print(params)

Xtrain = train_df.pipe(objectToCategory)[x_cols]
ytrain = train_df.pipe(objectToCategory)[y_col]

Xtest = test_df.pipe(objectToCategory)[x_cols]
ytest = test_df.pipe(objectToCategory)[y_col]

['funded_amnt', 'funded_amnt_inv', 'term', 'installment', 'grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'addr_state', 'dti', 'delinq_2yrs', 'fico_range_low', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status']
{'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 0.5, 'importance_type': 'split', 'learning_rate': 0.01, 'max_depth': 10, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 100, 'n_jobs': -1, 'num_leaves': 10, 'objective': 'binary', 'random_state': 42, 'reg_alpha': 0.0, 'reg_lambda': 0.05, 'subsample': 0.8, 'subsample_for_bin': 100, 'subsample_freq': 0, 'is_unbalance': False}


In [79]:
lgbm_model = lgbm.LGBMClassifier(**params)
lgbm_model.fit(
    Xtrain,
    ytrain,
    eval_set = [(Xtrain, ytrain), (Xtest, ytest)],
    eval_names = ['train', 'test'],
    eval_metric = 'auc'
)

[LightGBM] [Warning] Using too small ``bin_construct_sample_cnt`` may encounter unexpected errors and poor accuracy.
[LightGBM] [Info] Number of positive: 126319, number of negative: 493482
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.019681 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 333
[LightGBM] [Info] Number of data points in the train set: 619801, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.203806 -> initscore=-1.362676
[LightGBM] [Info] Start training from score -1.362676


LGBMClassifier(colsample_bytree=0.5, is_unbalance=False, learning_rate=0.01,
               max_depth=10, n_jobs=-1, num_leaves=10, objective='binary',
               random_state=42, reg_lambda=0.05, subsample=0.8,
               subsample_for_bin=100)

In [82]:
result = pd.DataFrame(lgbm_model.best_score_)
result = result.loc[['auc']].melt(value_name='auc')
result['gini'] = result['auc'] *2 -1
result

,variable,auc,gini
0,train,0.706730,0.413461
1,test,0.706421,0.412842


In [83]:
train_df['pred'] = lgbm_model.predict_proba(Xtrain)[:,1]
train_df[['default_flag', 'pred']].agg('mean')

default_flag    0.203806
pred            0.203859
dtype: float64

In [84]:
test_df['pred'] = lgbm_model.predict_proba(Xtest)[:,1]
test_df[['default_flag', 'pred']].agg('mean')

default_flag    0.202376
pred            0.203831
dtype: float64

In [86]:
joblib.dump(lgbm_model         , f'{output_path}lgbm_model_opt.gz')


['..\\..\\data\\outputs\\02_base_pipeline\\lgbm_model_opt.gz']